In [1]:
# Install and import libraries:
!pip install nilearn -q

from nilearn.interfaces.fmriprep import load_confounds_strategy, load_confounds
from nilearn.maskers import NiftiMasker
from nilearn import signal
from pathlib import Path
from tqdm.auto import tqdm
import nibabel as nib
import traceback

# Define root derivatives folder:
derivatives_dir = Path("../../dataset/derivatives")

# Create a text file to keep track of the already processed files:
processed_log = Path("cleaned_files.txt")

# Load already processed files if log exists:
if processed_log.exists():
    with open(processed_log, "r") as f:
        processed_files = set(line.strip() for line in f.readlines())
else:
    processed_files = set()

# Initialize a failed vector:
failed = []

# Find all preprocessed functional BOLD data:
bold_files = sorted(derivatives_dir.glob("sub-*/ses-*/func/*desc-preproc_bold.nii.gz"))

for bold_img_path in tqdm(bold_files, desc="Processing subjects"):

    # Convert path to string for logging/comparison:
    bold_str = str(bold_img_path)
    
    # Skip already processed files:
    if bold_str in processed_files:
        print(f"Skipping already processed filed: {bold_img_path.name}")
        continue

    try:
        # Define the corresponding brain mask file path:
        mask_path = Path(str(bold_img_path).replace("desc-preproc_bold.nii.gz", "desc-brain_mask.nii.gz"))

        # Create cleaned folder within the current subject's func/ folder:
        func_dir = bold_img_path.parent
        cleaned_dir = func_dir / "cleaned"
        cleaned_dir.mkdir(exist_ok=True)

        # Load image:
        img = nib.load(str(bold_img_path))
        mask = nib.load(str(mask_path))

        # Get FD-based scrubbing sample mask:
        _, sample_mask = load_confounds_strategy(
            str(bold_img_path),
            denoise_strategy="scrubbing",
            motion=None,
            scrub=0.5
        )

        # Obtain Global Signal Regression (GSR) confounds:
        gsr_confounds, _ = load_confounds(
            str(bold_img_path),
            strategy=["global_signal"],
            global_signal="basic"
        )

        # Extract BOLD signals with a masker:
        masker = NiftiMasker(mask_img=mask, standardize=False, detrend=False)
        raw_signals = masker.fit_transform(img)

        # Clean the signal:
        cleaned = signal.clean(
            raw_signals,
            confounds=gsr_confounds.to_numpy(),
            sample_mask=sample_mask,
            standardize="zscore_sample",
            detrend=True,
            high_pass=0.01,
            low_pass=0.1,
            t_r=img.header.get_zooms()[3]
        )

        # Reconstruct cleaned image:
        cleaned_img = masker.inverse_transform(cleaned)

        # Define output filename and path:
        output_filename = bold_img_path.name.replace("desc-preproc_bold", "desc-cleaned_scrubbed_gsr_bold")
        output_path = cleaned_dir / output_filename

        # Save cleaned image:
        cleaned_img.to_filename(str(output_path))

        # Append successfully processed file to log:
        with open(processed_log, "a") as f:
            f.write(bold_str + "\n")

        print(f"Processed: {bold_img_path.name}")

    except Exception as e:
        failed.append(str(bold_img_path))

        print(f"\nFAILED: {bold_img_path}")
        traceback.print_exc()

print("\n==============================")
print("PROCESSING FINISHED")
print("==============================")
print(f"Total files found: {len(bold_files)}")
print(f"Already processed skipped: {len(processed_files)}")
print(f"Failed files: {len(failed)}")

Processing subjects:   0%|          | 0/1188 [00:00<?, ?it/s]

Skipping already processed filed: sub-OAS30001_ses-d0129_task-rest_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS30002_ses-d0653_task-rest_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS30003_ses-d0558_task-rest_run-03_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS30004_ses-d1101_task-rest_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS30005_ses-d0581_task-rest_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS30006_ses-d1308_task-rest_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS30007_ses-d0061_task-rest_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS30008_ses-d0061_task-rest_run-02_space-MNI

/tmp/ipykernel_634/2040782023.py:71: UserWarning: [NiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  raw_signals = masker.fit_transform(img)
/tmp/ipykernel_634/2040782023.py:74: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned = signal.clean(


Processed: sub-OAS31209_ses-d0095_task-rest_run-03_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS31210_ses-d0064_task-restingstate_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS31211_ses-d0127_task-rest_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS31213_ses-d0114_task-rest_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS31214_ses-d0077_task-rest_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS31215_ses-d0063_task-restingstate_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS31217_ses-d0118_task-rest_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
Skipping already processed filed: sub-OAS31218_ses-d0067_task-restingstate_run-02_space-MN